# Week 3: Search Intelligence Data Contract — Content Refresh Engine

### 1. Data Contract Plain-Words Answers
* **Unit of Analysis (Grain):** One row represents exactly one `content_hash_id` aggregated over a temporal window.
* **Tables Used:** `fact_content_daily_performance` (GSC metrics) and `fact_content_metadata` (content properties).
* **Time Window:** Mid-panel historical baseline window (`report_date = '2026-03'`) paired with a subsequent evaluation window.
* **Target / Label Proxy:** `click_delta` (change in organic search clicks between windows). Negative values represent content decay.
* **Deliberate Exclusion:** Content updated within the last 14 days to prevent noise from ongoing re-indexing.

In [5]:
import os
import duckdb
import numpy as np
import pandas as pd

# Load Hugging Face token if available in environment secrets
hf_token = os.environ.get("HF_TOKEN", "")

# Generate synthetic mid-panel month dataset (2026-03)
np.random.seed(42)
dates = pd.date_range("2026-03-01", "2026-03-31")
content_ids = [f"hash_{i:04d}" for i in range(1, 101)]

records = []
for cid in content_ids:
    for d in dates:
        records.append({
            "content_hash_id": cid,
            "report_date": d,
            "gsc_impressions": np.random.randint(100, 1000),
            "gsc_clicks": np.random.randint(5, 100),
            "gsc_avg_position": np.round(np.random.uniform(1.0, 20.0), 1),
            "is_indexable": np.random.choice([True, False], p=[0.85, 0.15]),
            "word_count": np.random.randint(500, 3000)
        })

df_warehouse = pd.DataFrame(records)

# Connect DuckDB for fact verification queries
con = duckdb.connect()

# Query 1: Verify Grain
q1 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_ids
FROM (
    SELECT content_hash_id
    FROM df_warehouse
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY content_hash_id
);
"""
print("--- Query 1: Grain Verification ---")
display(con.sql(q1).df())

# Query 2: Slice Row Count & Date Span
q2 = """
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS total_daily_records
FROM df_warehouse
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31';
"""
print("\n--- Query 2: Date Span & Row Count ---")
display(con.sql(q2).df())

# Query 3: Availability Filtering (is_indexable IS TRUE)
q3 = """
SELECT
    COUNT(*) AS indexable_surviving_rows
FROM df_warehouse
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
  AND is_indexable IS TRUE;
"""
print("\n--- Query 3: Availability Check (IS TRUE) ---")
display(con.sql(q3).df())

--- Query 1: Grain Verification ---


,total_rows,unique_content_ids
0,100,100



--- Query 2: Date Span & Row Count ---


,min_date,max_date,total_daily_records
0,2026-03-01,2026-03-31,3100



--- Query 3: Availability Check (IS TRUE) ---


,indexable_surviving_rows
0,2658


In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 1. Build Feature Frame (Knowable Features)
# - mean_impressions: Knowable at decision time (historical metric)
# - mean_clicks: Knowable at decision time (historical metric)
# - mean_position: Knowable at decision time (historical metric)
# - word_count: Knowable at decision time (metadata spec)
# - ctr_baseline: Knowable at decision time (derived historical metric)

df_features = df_warehouse[df_warehouse["is_indexable"] == True].groupby("content_hash_id").agg(
    mean_impressions=("gsc_impressions", "mean"),
    mean_clicks=("gsc_clicks", "mean"),
    mean_position=("gsc_avg_position", "mean"),
    word_count=("word_count", "first")
).reset_index()

df_features["ctr_baseline"] = df_features["mean_clicks"] / df_features["mean_impressions"]

# Target Variable y
np.random.seed(42)
df_features["click_delta"] = np.random.normal(0, 15, size=len(df_features))

# 2. Add Leaked Column (The Trap)
df_features["LEAKED_future_clicks_proxy"] = df_features["click_delta"] * 0.99

X_honest = df_features[["mean_impressions", "mean_clicks", "mean_position", "word_count", "ctr_baseline"]]
X_leaked = df_features[["mean_impressions", "mean_clicks", "mean_position", "word_count", "ctr_baseline", "LEAKED_future_clicks_proxy"]]
y = df_features["click_delta"]

# Evaluate Models
m_honest = RandomForestRegressor(random_state=42).fit(X_honest, y)
m_leaked = RandomForestRegressor(random_state=42).fit(X_leaked, y)

print(f"MAE with Honest Features: {mean_absolute_error(y, m_honest.predict(X_honest)):.4f}")
print(f"MAE with LEAKED Feature (Trap): {mean_absolute_error(y, m_leaked.predict(X_leaked)):.4f}")

# Drop Leaked Column
df_features.drop(columns=["LEAKED_future_clicks_proxy"], inplace=True)
print("\n[SUCCESS] Leaked feature removed!")

MAE with Honest Features: 4.4361
MAE with LEAKED Feature (Trap): 0.2294

[SUCCESS] Leaked feature removed!


### 4. Named Limitation of the Slice
* **Seasonality & External Algorithmic Shifts:** Aggregating performance metrics over a single mid-panel month does not account for macro-seasonal traffic shifts or global Google Core Updates. A sudden rank drop may reflect algorithm re-indexing rather than true content obsolescence.

### 5. Self-Check Checklist
- [x] Contract answers provided in plain English.
- [x] Three verification queries executed with visible outputs (`IS TRUE` used for availability).
- [x] Five features defined with availability justification.
- [x] Deliberate leakage experiment run and leaked column dropped.
- [x] One slice limitation explicitly identified.